In [6]:
import pandas as pd
import pyreadstat
import os

# ==========================================
# 請確認您的 SAV 檔名是否正確
# ==========================================
sav_filename = r'C:\Users\user\Desktop\TIGPS_PLAN_DATA\TIGPS\2025_rawdata\changefile\TIGPS_W3_student.sav' 

if not os.path.exists(sav_filename):
    print(f"❌ 錯誤：找不到檔案 '{sav_filename}'，請確認檔名或路徑是否正確。")
else:
    print(f"✅ 找到檔案，開始讀取：{sav_filename} ...")

    try:
        # 1. 讀取數據 (套用數值標籤：1 -> 男)
        df, meta = pyreadstat.read_sav(sav_filename, apply_value_formats=True)
        print(f"✅ 讀取成功！資料共有 {df.shape[0]} 筆，{df.shape[1]} 個欄位。")

        # 2. 檢查是否有題目文字 (Variable Labels)
        if not meta.column_names_to_labels:
            print("⚠️ 警告：這個 SAV 檔案裡面似乎『沒有』儲存題目文字 (Variable Labels)！")
            print("   轉換後的 CSV 標題將維持 Q1, Q2 的代號。")
        else:
            print(f"✅ 檢測到題目文字字典，共 {len(meta.column_names_to_labels)} 個對應。")
            # 替換標題
            df = df.rename(columns=meta.column_names_to_labels)

        # 3. 存檔 (改用英文檔名以避免路徑錯誤)
        output_filename = 'VersionB_Full_Text.csv'
        
        print(f"⏳ 正在寫入 CSV 檔案: {output_filename} ...")
        df.to_csv(output_filename, index=False, encoding='utf-8-sig')

        # 4. 驗證檔案是否建立成功
        if os.path.exists(output_filename):
            file_size = os.path.getsize(output_filename)
            if file_size > 0:
                print(f"🎉 成功！檔案已建立：{output_filename}")
                print(f"📊 檔案大小：{file_size / 1024:.2f} KB")
                print("👉 請現在嘗試打開 'VersionB_Full_Text.csv' (不要打開中文檔名的舊檔案)")
            else:
                print(f"❌ 錯誤：檔案 {output_filename} 已建立，但是是空的 (0 KB)！")
        else:
            print(f"❌ 錯誤：無法建立檔案 {output_filename}")

    except Exception as e:
        print(f"❌ 發生未預期的錯誤：{e}")

❌ 錯誤：找不到檔案 'C:\Users\user\Desktop\TIGPS_PLAN_DATA\TIGPS\2025_rawdata\changefile\TIGPS_W3_student.sav'，請確認檔名或路徑是否正確。


In [7]:
import pandas as pd
import pyreadstat
import os

# 您的檔案路徑
sav_filename = r'C:\Users\user\Desktop\TIGPS_PLAN_DATA\TIGPS\2025_rawdata\changefile\TIGPS_W3_學生.sav'

print(f"🔄 正在讀取 SAV 檔案 (數值模式): {sav_filename} ...")

try:
    # 關鍵差異：這裡【不加】 apply_value_formats=True
    # 這樣讀出來的就會是原始的數字代碼 (1, 0, 99...)
    df, meta = pyreadstat.read_sav(sav_filename)
    
    print(f"✅ 讀取成功！資料共有 {df.shape[0]} 筆。")

    # 輸出檔名
    output_csv = 'VersionA_Statistical_Use.csv'
    output_xlsx = 'VersionA_Statistical_Use.xlsx'

    # 1. 存成 CSV (最通用)
    print(f"⏳ 正在寫入 CSV: {output_csv} ...")
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    
    # 2. 順便存成 Excel (方便查看，且數字格式比較不會跑掉)
    print(f"⏳ 正在寫入 Excel: {output_xlsx} ...")
    df.to_excel(output_xlsx, index=False, engine='openpyxl')

    print("\n🎉 轉換完成！")
    print(f"1. {output_csv} (適合匯入 SPSS/R/SAS)")
    print(f"2. {output_xlsx} (適合用 Excel 篩選檢查)")

except Exception as e:
    print(f"❌ 發生錯誤：{e}")

🔄 正在讀取 SAV 檔案 (數值模式): C:\Users\user\Desktop\TIGPS_PLAN_DATA\TIGPS\2025_rawdata\changefile\TIGPS_W3_學生.sav ...
✅ 讀取成功！資料共有 7714 筆。
⏳ 正在寫入 CSV: VersionA_Statistical_Use.csv ...
⏳ 正在寫入 Excel: VersionA_Statistical_Use.xlsx ...

🎉 轉換完成！
1. VersionA_Statistical_Use.csv (適合匯入 SPSS/R/SAS)
2. VersionA_Statistical_Use.xlsx (適合用 Excel 篩選檢查)


In [8]:
import pandas as pd
import numpy as np

# 1. 讀取您的 fulltext 檔案
input_csv = 'TIGPS_W3_student_convertedviacode_20260103_fulltext.csv'
print(f"🔄 正在讀取檔案：{input_csv} ...")
df = pd.read_csv(input_csv)

# 2. 定義完整的轉換字典 (Mapping Dictionary)
conversion_map = {
    # --- (A) 程度/符合度 (4=高) ---
    '很符合': 4, '非常同意': 4, '很同意': 4,
    '還算符合': 3, '還算同意': 3,
    '不太符合': 2, '不太同意': 2,
    '很不符合': 1, '非常不同意': 1, '很不同意': 1,

    # --- (B) 頻率 (4=高) ---
    '經常': 4, '總是': 4,
    '有時': 3,
    '偶爾': 2,
    '從未': 1,

    # --- (C) 是非/有無 (1=有, 0=無) ---
    '有': 1, '沒有': 0,
    '是': 1, '否': 0,
    '會': 1, '不會': 0,
    '沒有改變，原本就這樣': 0,  # Q12 特殊選項

    # --- (D) 性別 (1=男, 2=女) ---
    '男': 1, '男性': 1,
    '女': 2, '女性': 2,

    # --- (E) Q23 通訊頻率 (5=高) ---
    '每天好幾次': 5,
    '每天1次': 4,
    '每周3、4次': 3,
    '每周1、2次': 2,
    '幾乎沒有': 1,
    '無此人': 0,  # 設為0，分析時可視為不適用

    # --- (F) Q21 上網時間 (序數 1~10) ---
    '0.5小時以內': 1,
    '0.5-1小時': 2,
    '1-1.5小時': 3,
    '1.5-2小時': 4,
    '2-2.5小時': 5,
    '2.5-3小時': 6,
    '3-3.5小時': 7,
    '3.5-4小時': 8,
    '4-5小時': 9,
    '5小時以上': 10,

    # --- (G) Q9 成績 (1=最好) ---
    '全班五名以內': 1,
    '全班六至十名': 2,
    '全班十一至二十名': 3,
    '全班二十一名以後': 4,
    
    # --- (H) 睡眠品質 ---
    '非常好': 4,
    '還算好': 3,
    '不太好': 2,
    '非常不好': 1,
    
    # --- (I) 社會階層 (Q59) ---
    '最頂層': 10,
    '最底層': 1
}

print("🔄 正在執行轉換 (這可能需要幾秒鐘)...")

# 3. 執行全域替換
# 使用 replace 時設定 regex=False，確保完全符合文字才替換
# 這樣可以避免把句子中的「有」誤換成 1
df_numeric = df.replace(conversion_map)

# 4. 特殊處理：如果有文字沒被換掉，可以檢查一下
# 我們可以簡單統計一下現在還有多少欄位是 object (文字) 類型
non_numeric_cols = df_numeric.select_dtypes(include=['object']).columns
print(f"ℹ️ 轉換後仍保留文字格式的欄位 (通常是 ID 或開放式填答): {len(non_numeric_cols)} 個")

# 5. 存檔
output_csv = 'TIGPS_W3_Final_Numeric_Mapped.csv'
output_xlsx = 'TIGPS_W3_Final_Numeric_Mapped.xlsx'

print(f"⏳ 正在寫入 CSV: {output_csv} ...")
df_numeric.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f"⏳ 正在寫入 Excel: {output_xlsx} ...")
df_numeric.to_excel(output_xlsx, index=False, engine='openpyxl')

print("\n🎉 轉換成功！")
print("這份新檔案保留了【完整的中文題目標題】，但內容已變回【統計用的數字】。")
print("例如 Q4: '很符合' 已變成 4, '從未' 已變成 1。")

🔄 正在讀取檔案：TIGPS_W3_student_convertedviacode_20260103_fulltext.csv ...
🔄 正在執行轉換 (這可能需要幾秒鐘)...


C:\Users\user\AppData\Local\Temp\ipykernel_91988\2824487645.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_numeric = df.replace(conversion_map)


ℹ️ 轉換後仍保留文字格式的欄位 (通常是 ID 或開放式填答): 117 個
⏳ 正在寫入 CSV: TIGPS_W3_Final_Numeric_Mapped.csv ...
⏳ 正在寫入 Excel: TIGPS_W3_Final_Numeric_Mapped.xlsx ...

🎉 轉換成功！
這份新檔案保留了【完整的中文題目標題】，但內容已變回【統計用的數字】。
例如 Q4: '很符合' 已變成 4, '從未' 已變成 1。


In [ ]:
import pandas as pd
import numpy as np
import re

# 1. 讀取檔案
input_csv = 'TIGPS_W3_student_convertedviacode_20260103_fulltext.csv'
print(f"🔄 讀取檔案：{input_csv} ...")
df = pd.read_csv(input_csv)
df_new = df.copy()

# =================================================================
# 定義各類題型的轉換字典 (Mapping Dictionaries)
# =================================================================

# 1. 程度/同意度 (Likert Scale)
# -----------------------------------------------------------------
# Q4, Q6, Q11, Q25, Q26, Q29, Q43, Q61: 符不符合 (4點)
map_conformity_4pt = {
    '很符合': 4, '還算符合': 3, '不太符合': 2, '很不符合': 1,
    '不適用': 0, '沒使用過': 0
}

# Q27, Q28, Q52, Q53, Q60: 同意度 (4點)
map_agreement_4pt = {
    '很同意': 4, '還算同意': 3, '不太同意': 2, '很不同意': 1,
    '非常同意': 4, '非常不同意': 1 # 容錯
}

# Q40: 資訊素養 (出現了"同意", "非常同意") - 視為 5點量表
map_agreement_5pt = {
    '非常同意': 5, '很同意': 4, '同意': 3, '不太同意': 2, '非常不同意': 1,
    '很不同意': 1 # 容錯
}

# Q48: 科技課程 (4點)
map_tech_effect = {
    '很同意': 4, '還算同意': 3, '不太同意': 2, '很不同意': 1
}

# 2. 頻率題 (Frequency)
# -----------------------------------------------------------------
# Q5, Q19, Q24, Q31, Q33, Q35, Q37: 經常/從未
map_freq_4pt = {
    '經常': 4, '總是': 4, '有時': 3, '偶爾': 2, '從未': 1
}

# Q38: 旁觀霸凌 (多了一個"沒發生過")
map_freq_bystander = {
    '經常': 4, '有時': 3, '偶爾': 2, '從未': 1, '沒發生過': 0
}

# Q23: 通訊軟體頻率
map_comm_freq = {
    '每天好幾次': 6, '每天1次': 5, '每周3、4次': 4, '每周1、2次': 3, '幾乎沒有': 2, '無此人': 0
}

# Q41: AI使用頻率
map_ai_freq = {
    '每天': 5, '每週 5-6 次': 4, '每週 3-4 次': 3, '每週 1-2 次': 2, '從未': 1
}

# Q46: 老師教學頻率
map_teach_freq = {
    '幾乎每天教': 5, '一周1-2次': 4, '一個月1-2次': 3, '一學期1-2次': 2, '從未': 1
}

# Q54: 憂鬱量表 (時間)
map_depress_freq = {
    '最近兩週幾乎天天': 5, '最近1週五到七天': 4, '最近1週三到四天': 3, '最近1週一到兩天': 2, '完全沒有或少於一天': 1
}

# Q55: 幸福感 (比例)
map_happy_freq = {
    '大部分的時間': 5, '一半以上的時間': 4, '有時候': 3, '少於一半的時間': 2, '從來沒有': 1
}

# 3. 狀態/滿意度
# -----------------------------------------------------------------
# Q49, Q50, Q51, Q58-5: 滿意/快樂/健康/睡眠品質
map_status_4pt = {
    '很滿意': 4, '還算滿意': 3, '不太滿意': 2, '很不滿意': 1,
    '很快樂': 4, '還算快樂': 3, '不太快樂': 2, '很不快樂': 1,
    '很健康': 4, '還算健康': 3, '不太健康': 2, '很不健康': 1,
    '非常好': 4, '還算好': 3, '不太好': 2, '非常不好': 1 # 睡眠品質
}

# Q44: AI能力
map_ability = {
    '完全能做到': 4, '大部份能做到': 3, '能做到一些': 2, '完全做不到': 1
}

# 4. 是非/有無 (Binary)
# -----------------------------------------------------------------
# Q10, Q14, Q25-0, Q39, Q42, Q45, Q47
map_binary = {
    '有': 1, '會': 1, '是': 1, '曾經有': 1,
    '沒有': 0, '不會': 0, '否': 0, '不曾': 0,
    '不知道': 99 # Q41-1 知道/不知道
}

# 5. 特殊題組
# -----------------------------------------------------------------
# Q1: 性別
map_gender = {'男': 1, '女': 2}
# Q2: 性別認同
map_gender_id = {'男性': 1, '女性': 2, '其他': 3}
# Q3: 婚姻
map_marital = {
    '結婚，且同住一起': 1, '結婚，但未同住一起': 2, '離婚，但同住一起': 3, 
    '離婚，且分居': 4, '未婚，但同住一起': 5, '未婚，且分居': 6,
    '喪偶': 7, '親生父親過世': 8, '親生母親過世': 9, '其他（請說明）': 99
}
# Q9: 成績 (序數)
map_grades = {
    '全班五名以內': 1, '全班六至十名': 2, '全班十一至二十名': 3, 
    '全班二十一至三十名': 4, '全班三十名以後': 5
}
# Q12: 升學改變
map_change = {
    '有改變，比以前多': 3, '有改變，比以前少': 2, '沒有改變，原本就這樣': 1
}
# Q15-18: 學歷
map_edu = {
    '國中畢業': 1, '高中（職）畢業': 2, '專科畢業（五專、二專）': 3,
    '大學或技術學院畢業（四技、二技）': 4, '碩士畢業': 5, '博士畢業': 6,
    '我不打算繼續升學': 0, '軍校': 7, '公立高中': 2, '私立高中': 2, '高職': 2, '專科': 3,
    '不知道': 88, '其他，請說明': 99
}
# Q21: 上網時間
map_time_interval = {
    '0.5小時以內': 1, '0.5-1小時': 2, '1-1.5小時': 3, '1.5-2小時': 4,
    '2-2.5小時': 5, '2.5-3小時': 6, '3-3.5小時': 7, '3.5-4小時': 8,
    '4-4.5小時': 9, '4.5-5小時': 10, '5小時以上': 11, '沒有': 0
}
# Q62: 含糖飲料
map_drinks = {
    '沒有': 0, '1-7瓶': 1, '8-14瓶': 2, '15-21瓶': 3, '22-28': 4, '29-35瓶': 5, '36瓶以上': 6,
    '1-7瓶/罐': 1, '8-14瓶/罐': 2, '15-21瓶/罐': 3, '22-28瓶/罐': 4, '29-35瓶/罐': 5, '36瓶/罐以上': 6,
    '1-2杯': 1, '3-4杯': 2, '5-6杯': 3, '7-8杯': 4, '9-10': 5, '11杯以上': 6
}
# Q63: 咖啡
map_coffee = {
    '都沒有喝': 0, '一天不到1次': 1, '每天1次': 2, '每天2次': 3, 
    '每天3次': 4, '每天4次': 5, '每天5次（含）以上': 6
}
# Q58-1~4: 上午/下午
map_ampm = {'上午': 1, '下午': 2}

# =================================================================
# 設定題組對應規則 (Prefix -> Dictionary)
# =================================================================
# 這裡利用題號前綴 (如 "4.") 來指派對應的字典
group_mapping_rules = {
    '1.': map_gender,
    '2.': map_gender_id,
    '3.': map_marital,
    '4.': map_conformity_4pt,
    '5.': map_freq_4pt,
    '6.': map_conformity_4pt,
    '9.': map_grades,
    '10.': map_binary, '10-1.': map_binary,
    '11.': map_conformity_4pt,
    '12.': map_change,
    '13.': map_binary,
    '14.': map_binary, '14-1.': map_binary,
    '15.': map_edu, '16.': map_edu, '17.': map_edu, '18.': map_edu,
    '19-': map_freq_4pt,
    '21-': map_time_interval,
    '23.': map_comm_freq,
    '24.': map_freq_4pt,
    '25-0.': map_binary, '25.': map_conformity_4pt,
    '26.': map_conformity_4pt,
    '27.': map_agreement_4pt,
    '28.': map_agreement_4pt,
    '29.': map_conformity_4pt, # 注意: Q29用符不符合
    '30.': map_binary, '31.': map_freq_4pt,
    '32.': map_binary, '33.': map_freq_4pt,
    '34.': map_binary, '35.': map_freq_4pt,
    '36.': map_binary, '37.': map_freq_4pt,
    '38.': map_freq_bystander,
    '39.': map_binary,
    '40.': map_agreement_5pt, # 特殊 5點
    '41-1.': map_binary, '41.': map_ai_freq,
    '42.': map_binary,
    '43.': map_conformity_4pt,
    '44.': map_ability,
    '45.': map_binary,
    '46.': map_teach_freq,
    '47.': map_binary,
    '48.': map_tech_effect,
    '49.': map_status_4pt, '50.': map_status_4pt, '51.': map_status_4pt,
    '52.': map_agreement_4pt,
    '53.': map_agreement_4pt,
    '54.': map_depress_freq,
    '55.': map_happy_freq,
    '58-1.': map_ampm, '58-2.': map_ampm, '58-3.': map_ampm, '58-4.': map_ampm, '58-5.': map_status_4pt,
    '60.': map_agreement_4pt,
    '61.': map_conformity_4pt,
    '62-': map_drinks,
    '63.': map_coffee
}

# 輔助函數：取得欄位的前綴
def get_prefix(col_name):
    match = re.match(r'^(\d+[-_]?\d*)\.?', col_name)
    if match:
        return match.group(1) + "."
    return None

def safe_convert(value, mapper):
    if pd.isna(value): return value
    val_str = str(value).strip()
    
    # 1. 直接對應
    if val_str in mapper:
        return mapper[val_str]
    
    # 2. 如果原本就是數字 (例如填空題的 10, 24)，保留它
    try:
        return float(val_str)
    except ValueError:
        pass
        
    # 3. 都不符合 (例如 "其他：xxxx")，轉為 99
    return 99

print("🔄 開始依照題組規則轉換...")

# 遍歷所有欄位進行轉換
for col in df_new.columns:
    prefix = get_prefix(col)
    
    # 如果這個欄位有對應的規則
    if prefix and prefix in group_mapping_rules:
        mapper = group_mapping_rules[prefix]
        df_new[col] = df_new[col].apply(lambda x: safe_convert(x, mapper))
    
    # 特殊欄位 (ID 或 無前綴欄位) 不做處理，保持原樣

# =================================================================
# 存檔
# =================================================================
output_file = 'TIGPS_W3_Final_Numeric_Mapped.xlsx'
print(f"⏳ 正在寫入 Excel: {output_file} ...")
df_new.to_excel(output_file, index=False)

print(f"🎉 轉換完成！請檢查：{output_file}")